In [47]:
import torch
import random
import numpy as np
import pandas as pd 
from torch.utils.data import Dataset, DataLoader

In [48]:
# standardize function
def standardize(input_array):
    mean = np.nanmean(input_array)
    std = np.nanstd(input_array)
    return (input_array - mean) / std

In [49]:
#physio_path = '/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/physio_data_standardized_nonan.csv'
physio_path = '/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/physio_data_restandardized_cfc.csv'
physio_data = pd.read_csv(physio_path, index_col=False)
physio_data

,MouseID,Generation,SurvDays,Y1A_BW_BW,Y2A_BW_BW,Y3A_BW_BW,Y1_Glu.F_Glucose,Y2_Glu.F_Glucose,Y3_Glu.F_Glucose,Y1_Echo_BPM,...,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet,Mouse.ID,CFC.24mo.Average,cfc_bin
0,DO-40-2162,1.380871,2.687646,-0.383946,-0.782700,-1.196293,0.186248,-0.234392,-1.478529,-0.232497,...,0.211169,1.566909,-1.486526,-0.973168,-1.458577,1,40,DO-40-2162,3.78,0
1,DO-40-2179,1.380871,2.495570,0.718950,-1.072253,-1.202402,0.085515,-1.371614,-1.546019,-0.286371,...,0.491716,2.160584,0.428549,0.016183,-0.989636,1,40,DO-40-2179,26.71,1
2,DO-40-2181,1.380871,2.410203,-1.173090,-1.457490,-1.361323,0.085515,-0.987876,-1.657172,0.119472,...,0.961072,1.145936,-1.020848,0.016183,-1.225502,1,40,DO-40-2181,27.02,1
3,DO-40-2187,1.380871,2.363962,-0.151342,-0.590051,-0.847000,-0.828411,-1.127417,-1.521318,-0.765336,...,1.361592,2.160574,-0.551658,-0.559569,-1.107569,1,40,DO-40-2187,50.40,2
4,DO-40-2099,0.452363,2.783684,-0.490264,-1.069517,-1.235074,0.235828,-1.357252,-1.665074,-0.788309,...,0.328077,1.035564,-0.502281,-0.749367,-0.675501,1,40,DO-40-2099,34.18,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,DO-2D-4007,-1.404653,-0.300201,2.615591,0.920161,1.130443,-1.245374,0.323407,0.924049,0.662538,...,0.063930,-0.891058,1.927418,-0.212585,1.002585,3,2D,DO-2D-4007,23.69,1
500,DO-1D-3030,-1.404653,-0.382011,-0.476151,-0.533628,1.130443,-0.510581,-0.200526,0.924049,0.497200,...,-0.426312,-0.891058,1.927418,-1.488397,1.002585,4,1D,DO-1D-3030,29.78,1
501,DO-20-1021,-1.404653,-0.417580,0.792601,0.165291,1.130443,0.301472,0.090966,0.924049,0.468773,...,0.093096,-0.891058,1.927418,-0.058558,1.002585,2,20,DO-20-1021,18.22,0
502,DO-AL-0002,-1.404653,-0.357112,0.685565,1.078700,1.130443,-0.519040,-0.406045,0.924049,-0.588603,...,-0.868511,-0.891058,1.927418,0.481618,1.002585,5,AL,DO-AL-0002,57.42,2


In [51]:
np.min(physio_data.iloc[:, 1:-5]), np.max(physio_data.iloc[:, 1:-5])

(-5.397022540655006, 4.905619567652551)

In [52]:
np.mean(physio_data.iloc[:, 1:-5]), np.std(physio_data.iloc[:, 1:-5]).mean()

/home/rachel/miniforge3/envs/mm-vae/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


(0.0034791100989250852, 0.8265570750345952)

In [ ]:
pd.DataFrame(physio_var.values).to_csv('/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/physio_var.csv', index=False)

In [ ]:
len(physio_var)

36

In [30]:
genoprobs_path = '/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/genoprobs_rescaled.csv'
genoprobs = pd.read_csv(genoprobs_path)
genoprobs

,0,0.1,1,2,3,4,5,6,7,8,...,5863,5864,5865,5866,5867,5868,5869,5870,5871,5872
0,DO-1D-3001,0.885234,1.036678,1.103385,1.452176,-0.252292,1.184701,0.381302,-0.684006,1.182885,...,-0.771316,0.136388,0.174245,0.455512,-0.150298,-1.142339,0.021386,-0.992528,-2.699173,-1.535486
1,DO-1D-3002,-0.208122,1.919831,0.637959,0.430449,0.596863,0.231309,0.947508,0.365945,-0.476796,...,0.598006,0.041787,1.605438,-0.080431,0.585303,-1.128272,0.281775,-1.652954,1.004302,-0.040112
2,DO-1D-3003,1.156802,-0.055468,-2.028303,-1.758717,-1.483365,0.146188,0.546376,1.862347,-0.141052,...,0.304672,0.673915,0.573392,-0.321357,1.407647,-1.918260,-0.199639,2.141749,1.347697,-0.648107
3,DO-1D-3004,-0.528911,-0.671307,0.260803,0.335501,0.912233,1.995842,0.232460,0.426954,0.039277,...,0.514484,-0.161487,0.435819,0.890328,0.071865,-0.183059,-1.227350,-0.571656,0.158009,-1.191852
4,DO-1D-3005,-0.614594,0.549874,2.065360,0.615235,0.695816,0.120070,-0.551483,0.991362,-0.447724,...,0.927497,0.344556,-0.499409,-0.284400,0.633577,-1.155979,-2.188575,2.507976,-1.213588,1.625126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941,DO-AL-0035,-0.206126,0.333981,0.423745,-0.938993,1.461668,0.116169,0.714316,1.247471,-0.904540,...,0.749489,-0.495239,0.104451,-0.386706,0.255033,-0.438960,0.064684,-1.259502,-0.308734,-0.705809
942,DO-AL-0064,-1.313605,-1.092530,-0.231617,0.543530,1.860594,-1.862789,-0.564824,0.247141,-1.651349,...,-1.929789,1.023983,3.292974,-1.149786,0.417695,1.180883,-1.169043,-0.292765,0.363160,0.512706
943,DO-AL-0073,3.000092,-0.726186,0.754349,0.957058,-1.486071,0.499843,-0.182949,-0.174342,-0.268673,...,2.375086,1.578232,1.471247,1.086538,0.745020,1.584566,2.043848,0.526358,-0.332411,0.607431
944,DO-AL-0089,-0.736350,-0.595276,-0.759997,0.033345,2.164433,-1.678936,0.914561,1.865717,0.813978,...,-1.071632,-0.041443,0.082546,0.548772,-2.751350,0.568229,1.562883,0.869626,0.216026,-1.873143


In [32]:
np.min(genoprobs.iloc[:, 1:]), np.max(genoprobs.iloc[:, 1:])

(-5.626699932437837, 5.407817280582792)

In [33]:
np.mean(genoprobs.iloc[:, 1:])

-3.09815293583609e-19

In [34]:
genoprobs_var = genoprobs.iloc[:, 1:].var(axis=0)
genoprobs_var

0.1     1.001058
1       1.001058
2       1.001058
3       1.001058
4       1.001058
          ...   
5868    1.001058
5869    1.001058
5870    1.001058
5871    1.001058
5872    1.001058
Length: 5873, dtype: float64

In [ ]:
pd.DataFrame(genoprobs_var.values).to_csv('/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/genoprobs_var.csv', index=False)

In [35]:
genoprobs = genoprobs.rename(columns={genoprobs.columns[0]: 'MouseID'})
genoprobs

,MouseID,0.1,1,2,3,4,5,6,7,8,...,5863,5864,5865,5866,5867,5868,5869,5870,5871,5872
0,DO-1D-3001,0.885234,1.036678,1.103385,1.452176,-0.252292,1.184701,0.381302,-0.684006,1.182885,...,-0.771316,0.136388,0.174245,0.455512,-0.150298,-1.142339,0.021386,-0.992528,-2.699173,-1.535486
1,DO-1D-3002,-0.208122,1.919831,0.637959,0.430449,0.596863,0.231309,0.947508,0.365945,-0.476796,...,0.598006,0.041787,1.605438,-0.080431,0.585303,-1.128272,0.281775,-1.652954,1.004302,-0.040112
2,DO-1D-3003,1.156802,-0.055468,-2.028303,-1.758717,-1.483365,0.146188,0.546376,1.862347,-0.141052,...,0.304672,0.673915,0.573392,-0.321357,1.407647,-1.918260,-0.199639,2.141749,1.347697,-0.648107
3,DO-1D-3004,-0.528911,-0.671307,0.260803,0.335501,0.912233,1.995842,0.232460,0.426954,0.039277,...,0.514484,-0.161487,0.435819,0.890328,0.071865,-0.183059,-1.227350,-0.571656,0.158009,-1.191852
4,DO-1D-3005,-0.614594,0.549874,2.065360,0.615235,0.695816,0.120070,-0.551483,0.991362,-0.447724,...,0.927497,0.344556,-0.499409,-0.284400,0.633577,-1.155979,-2.188575,2.507976,-1.213588,1.625126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941,DO-AL-0035,-0.206126,0.333981,0.423745,-0.938993,1.461668,0.116169,0.714316,1.247471,-0.904540,...,0.749489,-0.495239,0.104451,-0.386706,0.255033,-0.438960,0.064684,-1.259502,-0.308734,-0.705809
942,DO-AL-0064,-1.313605,-1.092530,-0.231617,0.543530,1.860594,-1.862789,-0.564824,0.247141,-1.651349,...,-1.929789,1.023983,3.292974,-1.149786,0.417695,1.180883,-1.169043,-0.292765,0.363160,0.512706
943,DO-AL-0073,3.000092,-0.726186,0.754349,0.957058,-1.486071,0.499843,-0.182949,-0.174342,-0.268673,...,2.375086,1.578232,1.471247,1.086538,0.745020,1.584566,2.043848,0.526358,-0.332411,0.607431
944,DO-AL-0089,-0.736350,-0.595276,-0.759997,0.033345,2.164433,-1.678936,0.914561,1.865717,0.813978,...,-1.071632,-0.041443,0.082546,0.548772,-2.751350,0.568229,1.562883,0.869626,0.216026,-1.873143


In [38]:
physio_data['MouseID'].isin(genoprobs['MouseID']).sum()

929

In [39]:
physio_data['MouseID'].dtypes

dtype('O')

In [ ]:
# physio_data['MouseID'] = physio_data['MouseID'].astype(str)
# genoprobs['MouseID'] = genoprobs['MouseID'].astype(str)

In [ ]:
genoprobs['MouseID'].dtypes

dtype('O')

In [ ]:
physio_data['MouseID'].map(type).unique()

array([<class 'str'>], dtype=object)

In [ ]:
genoprobs['MouseID'].map(type).unique()

array([<class 'str'>], dtype=object)

In [56]:
joined_genetic_physio = pd.merge(genoprobs, physio_data, on='MouseID', how='inner')
joined_genetic_physio

,MouseID,0.1,1,2,3,4,5,6,7,8,...,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet,Mouse.ID,CFC.24mo.Average,cfc_bin
0,DO-1D-3001,0.885234,1.036678,1.103385,1.452176,-0.252292,1.184701,0.381302,-0.684006,1.182885,...,-0.011158,0.770557,1.927418,-0.322225,-1.087006,4,1D,DO-1D-3001,0.84,0
1,DO-1D-3003,1.156802,-0.055468,-2.028303,-1.758717,-1.483365,0.146188,0.546376,1.862347,-0.141052,...,-0.411678,0.293438,1.927418,-0.403810,-1.019388,4,1D,DO-1D-3003,83.69,4
2,DO-1D-3010,0.147212,-0.629090,0.095689,-0.822066,-1.147267,0.014700,0.237685,-1.764237,-1.134055,...,-0.305292,0.824915,1.927418,-0.403810,-1.373187,4,1D,DO-1D-3010,36.71,1
3,DO-1D-3011,0.501676,-1.656495,0.752401,-0.348172,1.043520,-0.411199,1.418046,-1.336511,-0.988819,...,0.057679,1.283924,1.927418,-0.595036,-0.783522,4,1D,DO-1D-3011,21.96,1
4,DO-1D-3013,-1.894295,-0.589873,0.736833,0.101999,-0.042866,0.525641,-0.812557,0.540155,0.514178,...,-0.217680,-0.891058,1.927418,-0.364648,1.002585,4,1D,DO-1D-3013,6.93,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,DO-AL-0186,-0.408746,0.723953,-0.114496,0.852664,-0.156098,-0.293482,0.724477,-1.353684,-0.640700,...,0.128756,-0.891058,-0.296149,-0.939946,1.002585,5,AL,DO-AL-0186,20.44,1
497,DO-AL-0188,0.885158,-0.713064,0.707135,-0.531217,1.003639,1.606643,0.987814,-0.958576,-1.891393,...,0.247655,0.910384,-0.779282,-0.175043,-1.225502,5,AL,DO-AL-0188,18.31,0
498,DO-AL-0191,0.269565,0.035678,-0.173052,-1.004266,-1.580964,0.475946,0.805426,0.353003,0.781255,...,-0.109053,-0.891058,0.048480,0.288994,1.002585,5,AL,DO-AL-0191,24.71,1
499,DO-1D-3083,-1.650913,0.649956,-0.438226,0.676653,-0.926468,1.070770,0.104359,0.712658,1.142204,...,1.192684,1.821772,-1.509398,-0.981960,-1.069439,4,1D,DO-1D-3083,31.07,1


In [57]:
joined_genetic_physio.iloc[:, 1:5874]

,0.1,1,2,3,4,5,6,7,8,9,...,5863,5864,5865,5866,5867,5868,5869,5870,5871,5872
0,0.885234,1.036678,1.103385,1.452176,-0.252292,1.184701,0.381302,-0.684006,1.182885,1.393060,...,-0.771316,0.136388,0.174245,0.455512,-0.150298,-1.142339,0.021386,-0.992528,-2.699173,-1.535486
1,1.156802,-0.055468,-2.028303,-1.758717,-1.483365,0.146188,0.546376,1.862347,-0.141052,-1.221509,...,0.304672,0.673915,0.573392,-0.321357,1.407647,-1.918260,-0.199639,2.141749,1.347697,-0.648107
2,0.147212,-0.629090,0.095689,-0.822066,-1.147267,0.014700,0.237685,-1.764237,-1.134055,-0.308885,...,0.764198,-0.587157,0.305768,-0.711702,-0.086673,-0.421039,-0.448817,0.720776,0.286950,-0.031061
3,0.501676,-1.656495,0.752401,-0.348172,1.043520,-0.411199,1.418046,-1.336511,-0.988819,0.733376,...,1.117912,-0.000047,-2.660262,1.029520,-0.561236,-0.604382,0.748423,0.679467,-0.633292,-0.884505
4,-1.894295,-0.589873,0.736833,0.101999,-0.042866,0.525641,-0.812557,0.540155,0.514178,-2.245385,...,1.011583,-1.374545,0.343487,-2.378116,-0.078883,0.660231,-1.179689,1.189737,-0.002943,-0.561462
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,-0.408746,0.723953,-0.114496,0.852664,-0.156098,-0.293482,0.724477,-1.353684,-0.640700,-0.048372,...,0.610166,0.147378,0.463245,-1.462084,0.376246,-0.798738,-0.951866,1.020758,0.439428,0.337400
497,0.885158,-0.713064,0.707135,-0.531217,1.003639,1.606643,0.987814,-0.958576,-1.891393,0.399979,...,-0.803410,0.530663,-0.615658,0.774813,0.041362,-1.477720,-0.856324,-0.833098,-0.401328,0.511024
498,0.269565,0.035678,-0.173052,-1.004266,-1.580964,0.475946,0.805426,0.353003,0.781255,-0.788710,...,-0.942568,1.197563,-1.201868,-0.251591,0.280121,-0.770785,-0.119193,-1.420337,0.177190,0.766062
499,-1.650913,0.649956,-0.438226,0.676653,-0.926468,1.070770,0.104359,0.712658,1.142204,-1.667080,...,-0.700140,-2.102677,-2.179107,0.161074,-1.299613,-0.081531,-0.871043,-1.003408,0.207623,0.456482


In [59]:
joined_genetic_physio.iloc[:, 5874:-5]

,Generation,SurvDays,Y1A_BW_BW,Y2A_BW_BW,Y3A_BW_BW,Y1_Glu.F_Glucose,Y2_Glu.F_Glucose,Y3_Glu.F_Glucose,Y1_Echo_BPM,Y2_Echo_BPM,...,Y3_Wheel_AvgSpeedLFC,Y1_Wheel_AvgDistLFC,Y2_Wheel_AvgDistLFC,Y3_Wheel_AvgDistLFC,Y1A_Grip_All,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj
0,-1.404653,0.140862,0.585946,0.203613,-0.554682,-0.875993,-0.513840,-0.967201,-0.094961,-1.532842,...,-0.527638,-0.766422,0.562758,-0.521205,-1.836728,-0.011158,0.770557,1.927418,-0.322225,-1.087006
1,-1.404653,0.247571,-1.412499,-1.069015,-1.066964,0.179382,0.096653,-0.263232,0.343306,-2.215901,...,-0.527638,-0.105768,0.824053,-0.521205,-0.861098,-0.411678,0.293438,1.927418,-0.403810,-1.019388
2,-1.404653,1.104799,-1.804734,-1.084319,-1.008261,-0.348305,-0.147544,-0.757246,-0.039118,-1.128095,...,1.896406,0.893294,1.206474,2.567578,-0.504917,-0.305292,0.824915,1.927418,-0.403810,-1.373187
3,-1.404653,0.663736,-0.012402,-0.086052,-0.552581,-0.031693,-0.199872,-0.090327,0.500020,0.303138,...,-0.527638,0.889320,0.911834,-0.521205,-1.222449,0.057679,1.283924,1.927418,-0.595036,-0.783522
4,-1.404653,-0.157922,0.495062,-0.128831,-0.929132,-0.348305,-0.444070,0.924049,-0.427521,-1.061179,...,-0.527638,-0.532866,0.671313,-0.521205,-1.588946,-0.217680,-0.891058,1.927418,-0.364648,1.002585
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,1.380871,-0.428251,0.165585,0.274891,1.130443,-1.049013,0.459865,0.924049,0.467919,-0.202692,...,-0.527638,0.167509,0.368243,-0.521205,-0.267554,0.128756,-0.891058,-0.296149,-0.939946,1.002585
497,1.380871,0.418305,-1.102426,-0.071373,-0.411818,-0.046407,-0.237841,-1.064356,-0.202265,-0.068778,...,-0.527638,-0.094153,0.520245,-0.521205,-1.197889,0.247655,0.910384,-0.779282,-0.175043,-1.225502
498,1.380871,-0.083226,0.135917,0.436642,-0.151508,-0.310251,-0.011087,0.924049,-0.849113,-0.769359,...,-0.527638,0.005343,0.229820,-0.521205,0.835847,-0.109053,-0.891058,0.048480,0.288994,1.002585
499,-0.476145,0.574812,0.770326,0.402476,-0.107329,0.423583,1.957102,-0.099684,-0.212584,-0.829955,...,-0.527638,-0.526726,1.123056,-0.521205,1.273271,1.192684,1.821772,-1.509398,-0.981960,-1.069439


In [60]:
#joined_genetic_physio.to_csv('/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/joined_genetic_physio_restandardized.csv', index=False)
joined_genetic_physio.to_csv('/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/joined_genetic_physio_restandardized_cfc.csv', index=False)

In [ ]:
joined_genetic_physio = joined_genetic_physio.drop('Diet_y', axis=1)
joined_genetic_physio = joined_genetic_physio.rename(columns={'Diet_x': 'Diet'})

In [ ]:
joined_genetic_physio

,MouseID,0,1,2,3,4,5,6,7,8,...,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet,Mouse.ID,CFC.24mo.Average,cfc_bin
0,DO-1D-3001,11.181311,12.383703,13.223754,18.683861,-3.105873,15.071812,4.566722,-8.317668,15.178740,...,-0.282162,-0.284390,-0.302189,-0.302202,-0.302199,4,1D,DO-1D-3001,0.84,0
1,DO-1D-3003,14.611459,-0.662600,-24.308638,-22.627848,-18.261186,1.859805,6.543754,22.646550,-1.809972,...,-0.284586,-0.287381,-0.302189,-0.302203,-0.302198,4,1D,DO-1D-3003,83.69,8
2,DO-1D-3010,1.859422,-7.514837,1.146809,-10.576791,-14.123598,0.187016,2.846669,-21.453514,-14.552152,...,-0.283942,-0.284049,-0.302189,-0.302203,-0.302205,4,1D,DO-1D-3010,36.71,3
3,DO-1D-3011,6.336620,-19.787761,9.017313,-4.479619,12.846408,-5.231286,16.983432,-16.252265,-12.688484,...,-0.281745,-0.281171,-0.302189,-0.302205,-0.302193,4,1D,DO-1D-3011,21.96,2
4,DO-1D-3013,-23.926661,-7.046359,8.830735,1.312327,-0.527705,6.687224,-9.731708,6.568401,6.597907,...,-0.283412,-0.294808,-0.302189,-0.302202,-0.302156,4,1D,DO-1D-3013,6.93,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,DO-AL-0186,-5.162836,8.648018,-1.372207,10.970472,-1.921667,-3.733688,8.676810,-16.461100,-8.221440,...,-0.281315,-0.294808,-0.302211,-0.302210,-0.302156,5,AL,DO-AL-0186,20.44,2
497,DO-AL-0188,11.180352,-8.517944,8.474807,-6.834697,12.355449,20.439765,11.830693,-11.656498,-24.270286,...,-0.280596,-0.283513,-0.302216,-0.302200,-0.302202,5,AL,DO-AL-0188,18.31,1
498,DO-AL-0191,3.404846,0.426192,-2.073981,-12.920995,-19.462683,6.055004,9.646308,4.292594,10.025039,...,-0.282754,-0.294808,-0.302207,-0.302194,-0.302156,5,AL,DO-AL-0191,24.71,2
499,DO-1D-3083,-20.852529,7.764088,-5.252014,8.705888,-11.405415,13.622378,1.249871,8.666080,14.656721,...,-0.274878,-0.277799,-0.302223,-0.302210,-0.302199,4,1D,DO-1D-3083,31.07,3


In [ ]:
joined_genetic_physio.to_csv('/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/joined_genetic_physio_cfc.csv', index=False)

In [ ]:
joined_genetic_physio = pd.read_csv('/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/joined_genetic_physio_cfc.csv')

In [ ]:
joined_genetic_physio.iloc[:, 1:genetic_dims]

NameError: name 'genetic_dims' is not defined

In [ ]:
len(joined_genetic_physio.iloc[:, 5874:-4].columns)

36

In [ ]:
class PairedDODataset(Dataset):
    def __init__(self, paired_df, genetic_dims, diet_col, id_col, label_col): 
        self.paired_df = paired_df
        self.genetic_dims = genetic_dims
        self.diet_col = diet_col 
        self.id_col = id_col
        self.label_col = label_col

    def __len__(self):
        return len(self.paired_df)

    def __getitem__(self, idx):
        sample = self.paired_df.iloc[idx]

        # for the genoprobs data 
        genetic_df = torch.tensor(sample[1:self.genetic_dims].values.astype('float32'))
        genetic_label = sample[self.label_col]
        genetic_diet = sample[self.diet_col]
        genetic_id = sample[self.id_col]

        # for the physiological data 
        physio_df = torch.tensor(sample[self.genetic_dims:-2].values.astype('float32'))
        physio_label = sample[self.label_col]
        physio_diet = sample[self.diet_col]
        physio_id = sample[self.id_col]

        return (genetic_df, genetic_label, genetic_diet, genetic_id), (physio_df, physio_label, physio_diet, physio_id)

In [ ]:
genetic_dims = 5874
diet_col = 'Diet'
id_col = 'MouseID'
label_col = 'Diet_num'
dataset = PairedDODataset(joined_genetic_physio, genetic_dims, diet_col, id_col, label_col)

In [ ]:
dataloader = DataLoader(dataset, batch_size=512, shuffle=True)

In [ ]:
first_iter = next(iter(dataloader))

In [ ]:
first_iter[0][0]

In [ ]:
first_iter[1][0].shape

In [ ]:
len(first_iter[1][3])